<a href="https://colab.research.google.com/github/Afzal498/Final-Exam_Q2_irfan_afzal/blob/Agentpro/Final/Agent_pro_My_Healthy_Way_of_life_Q2_Final_ExamF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# 🏋️‍♀️ Healthy Lifestyle Assistant 🥗
# **No API Keys Needed - 100% Free Version**

# %% [code]
# Install required packages
!pip install transformers torch accelerate sentencepiece bitsandbytes termcolor

# %% [code]
import json
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from termcolor import colored
from IPython.display import clear_output

# %% [code] {type: "setup"}
# Select a free local AI model
MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"  # Lightweight but powerful

# Load model with 4-bit quantization to save memory
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create text generation pipeline
health_llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7
)

print(colored("✓ Loaded free AI model successfully!", "green"))

# %% [code] {type: "tools"}
class NutritionTool:
    def __init__(self):
        self.name = "NutritionAnalyzer"
        self.description = "Analyzes nutrition information for food items"
        self.nutrition_db = {
            "egg": {"calories": 70, "protein": 6, "carbs": 0.6, "fat": 5},
            "apple": {"calories": 95, "protein": 0.5, "carbs": 25, "fat": 0.3},
            "chicken breast": {"calories": 165, "protein": 31, "carbs": 0, "fat": 3.6},
            "salad": {"calories": 50, "protein": 2, "carbs": 8, "fat": 1},
            "banana": {"calories": 105, "protein": 1.3, "carbs": 27, "fat": 0.4},
            "rice": {"calories": 130, "protein": 2.7, "carbs": 28, "fat": 0.3},
            "bread": {"calories": 80, "protein": 3, "carbs": 15, "fat": 1}
        }

    def run(self, query):
        # Simple NLP to match food items
        query = query.lower()
        for food, data in self.nutrition_db.items():
            if food in query:
                return data

        # Estimate for unknown items
        return {
            "calories": 150,
            "protein": 10,
            "carbs": 20,
            "fat": 5,
            "note": "Estimated values"
        }

class ExercisePlanner:
    def __init__(self):
        self.name = "ExercisePlanner"
        self.description = "Generates personalized workout plans"

    def run(self, fitness_level, goal):
        workouts = {
            "beginner": {
                "weight_loss": [
                    "5 min warm-up (arm circles, leg swings)",
                    "20 min brisk walking or light jogging",
                    "5 min cool-down stretching"
                ],
                "muscle_gain": [
                    "5 min warm-up",
                    "3 sets of 10 push-ups",
                    "3 sets of 10 bodyweight squats",
                    "3 sets of 10 lunges (each leg)",
                    "5 min cool-down"
                ]
            },
            "intermediate": {
                "weight_loss": [
                    "5 min warm-up",
                    "15 min HIIT: 30s sprint/30s rest",
                    "10 min strength circuit (squats, push-ups, lunges)",
                    "5 min cool-down"
                ],
                "muscle_gain": [
                    "5 min warm-up",
                    "4 sets of 10 dumbbell presses",
                    "4 sets of 10 bent-over rows",
                    "4 sets of 15 squats",
                    "5 min cool-down"
                ]
            }
        }
        return workouts.get(fitness_level, {}).get(goal, ["No workout plan found"])

class ProgressTracker:
    def __init__(self):
        self.name = "ProgressTracker"
        self.description = "Tracks and manages user health metrics"
        self.user_data = {}

    def run(self, user_id, action, data=None):
        try:
            if user_id not in self.user_data:
                self.user_data[user_id] = {}

            if action == "save":
                self.user_data[user_id].update(data)
                return {"status": "success", "message": "Data saved"}

            elif action == "get":
                return self.user_data.get(user_id, {})

            return {"error": "Invalid action"}
        except Exception as e:
            return {"error": str(e)}

# %% [code] {type: "agent"}
class AgentPro:
    def __init__(self):
        self.tools = {
            "nutrition": NutritionTool(),
            "exercise": ExercisePlanner(),
            "progress": ProgressTracker()
        }
        self.system_prompt = """
        You are FitBot, a friendly Healthy Lifestyle Assistant. Help users with nutrition,
        exercise, and health tracking using these tools when needed:

        Tools:
        1. NutritionAnalyzer - for food nutrition facts (input: food description)
        2. ExercisePlanner - creates workouts (input: fitness_level, goal)
        3. ProgressTracker - manages health data (input: user_id, action, data)

        Rules:
        - ALWAYS use tools for calculations and data retrieval
        - For ProgressTracker:
            action: 'save' or 'get'
            data: dictionary like {"weight": 70}
        - Keep responses concise but helpful
        - Use metric system (kg, cm, etc.)
        - Add relevant emojis 🥗💪📊
        """

    def process(self, user_input, user_id="default_user"):
        # Create the initial prompt
        prompt = f"""
        <|system|>
        {self.system_prompt}</s>
        <|user|>
        User ID: {user_id}
        Query: {user_input}</s>
        <|assistant|>
        """

        # First pass: Determine if tool is needed
        response = health_llm(
            prompt,
            max_new_tokens=200,
            stop_sequence="</s>"
        )[0]['generated_text']

        # Extract tool call if present
        tool_call = None
        if "Tool:" in response:
            try:
                tool_part = response.split("Tool:")[1].strip()
                tool_name = tool_part.split("(")[0].strip()
                params_str = tool_part.split("(")[1].split(")")[0].strip()

                # Simple parameter parsing
                params = {}
                for pair in params_str.split(","):
                    key, value = pair.split(":")
                    params[key.strip()] = value.strip().strip('"')

                tool_call = (tool_name, params)
            except:
                pass

        # If tool is called
        if tool_call:
            tool_name, params = tool_call
            print(colored(f"\n🔧 Using tool: {tool_name} with params: {params}", "magenta"))

            # Execute tool
            tool = self.tools.get(tool_name.lower())
            if not tool:
                return "Error: Tool not found"

            # Add user_id for progress tracker
            if tool_name == "ProgressTracker" and "user_id" not in params:
                params["user_id"] = user_id

            tool_response = tool.run(**params)

            # Create new prompt with tool response
            new_prompt = f"""
            {prompt}{response}
            Tool Result: {json.dumps(tool_response)}
            """

            # Generate final response
            final_response = health_llm(
                new_prompt,
                max_new_tokens=400,
                stop_sequence="</s>"
            )[0]['generated_text']

            # Extract only the assistant's response
            return final_response.split("<|assistant|>")[-1].strip()

        return response.split("<|assistant|>")[-1].strip()

# %% [code] {type: "main"}
def main():
    clear_output()
    print(colored("\n" + "="*50, "blue"))
    print(colored("🏋️‍♀️ HEALTHY LIFESTYLE ASSISTANT (FREE VERSION) 🥗", "green", attrs=["bold"]))
    print(colored("="*50, "blue"))

    agent = AgentPro()
    user_id = input("\nEnter your name: ").strip() or "guest"

    print("\nHow can I help you today? (Examples below)")
    print(colored("• 'I ate 2 eggs and an apple'", "yellow"))
    print(colored("• 'Create a beginner workout for weight loss'", "yellow"))
    print(colored("• 'Save my weight as 75kg'", "yellow"))
    print(colored("• 'Show my progress'", "yellow"))
    print(colored("• Type 'exit' to quit\n", "yellow"))

    while True:
        try:
            query = input(colored("\nYou: ", "yellow"))
            if query.lower() in ["exit", "quit"]:
                print(colored("\n👋 Have a healthy day! Goodbye!", "green"))
                break

            response = agent.process(query, user_id)
            print(colored(f"\nFitBot: {response}", "cyan"))

        except KeyboardInterrupt:
            print(colored("\n👋 Session ended. Stay healthy!", "green"))
            break
        except Exception as e:
            print(colored(f"\n🚨 Error: {str(e)}", "red"))

# %% [code] {type: "run"}
if __name__ == "__main__":
    main()


🏋️‍♀️ HEALTHY LIFESTYLE ASSISTANT (FREE VERSION) 🥗

Enter your name: irfan

How can I help you today? (Examples below)
• 'I ate 2 eggs and an apple'
• 'Create a beginner workout for weight loss'
• 'Save my weight as 75kg'
• 'Show my progress'
• Type 'exit' to quit


You: what are the nutritions in 4  eggs

FitBot: 🍳 NutritionAnalyzer says:

        4 large eggs provide approximately:

        - Calories: 196 kcal (815 kJ)
        - Protein: 12.6 g
        - Fat: 11.6 g (16% of daily value)
        - Carbohydrates: 0.6 g
        - Fiber: 0.2 g
        - Sugar: 0.5 g
        - Sodium: 510 mg (21% of daily value)

        Note: The nutritional value may vary depending on factors such as the eggs' origin and the yolks' size.
        😊 Remember, eggs are a great source of protein, so they can help you build muscle and power through your day! 💪

You: my age is 44. give my workout plan

FitBot: 🤝 Hey, Irfan! Based on your age of 44, here's a workout plan created by ExercisePlanner for you:

